In [ ]:
import json

from dotenv import load_dotenv
from google.genai import Client

from src.dto.transaction_dto import Transaction, TransactionsList

In [ ]:
load_dotenv()

In [ ]:
system_prompt = """
You are a data synthesizing agent for a financial application's NLP pipeline. The pipeline requires training data that will be used to train a blank model. Your task is to generate 50 SMS texts with their metadata following these instructions:

## Instructions:
### 1. Text generation:
- The length of the text should lie between 15-50 words.
- Occasionally include noise such as double-spacing, abbreviations and missing punctuations to allow the model understand various text format.
- Generate with varying sentence structure such as shifting the location of where the transaction code or balance is.
- DO NOT intentionally introduce grammatical errors such as spelling errors or cutting off words to make noise.
- Optionally append unnecessary information such as adverts, awards, notices at the end of the main content.
- Financial digits can include commas as the separator within the text and not the metadata.
- **Transaction code:**
      - Each text MUST have a transactional code if the text indicates the money is going outwards from the account.
      - A transaction code is characterized by a word with alphanumeric charcters with a minimum of 10 characters with letters being in UPPERCASE.
      - The code should be placed anywhere within the text.
      - Occasionally include abbreviations such as Ref, txn, Confirmed and any other text that can be used to indicate that a transactional code follows it.
      - Occasionally include two different codes in one text. For example: 'Confirmed. F8SFD6FW95 successfully withdrawn from ABSA BANK account 1234. M-Pesa ref. 5SRE6WE68F'. In such cases, the first reference is the desired one
      - Some examples of how codes look like are FS4F6R68WREW9, WERW9849FD, WEW684D68F4. You aren't restricted to these alone.
- **Vendor:**
      - Optionally include an organization/sacco/company/bank name as the vendor.
      - Vendors are simply where the recipient account's lives.
      - The vendor's name should be in UPPERCASE.
      - Not every text necessarily requires a vendor, some recipients don't use vendors.
      - Examples of vendors can include ABSA BANK, KCB BANK, EQUITY, SOME ORG. You can generate other vendor names.
- **Recipient:**
      - This is the account actually receiving the cash.
      - A recipient can be a person's name, business name, account number or phone number.
      - This is an optional field but should be included in some of the texts.
      - If a vendor is provided, it should be close to the vendors name for example, 'Successfully sent Ksh. 34.00 to John F. Kennedy at ABSA BANK', 'Ksh. 50.00 sent to JAMII TELECOMMUNICATIONS for account +254712345678-fh5'.
- **Action:**
      - This is the text determining what's actually happening during the transaction.
      - Examples (not limited) include deposited, sent to, withdrawn, paid to, reversed, transacted, etc.
      - This MUST be included.
- **Date and Time:**
      - MUST be included.
      - Dates can be separated using '/' or '-'. The standard format is day(dd), month(mm) and year(yyyy) but you can include other formats for the noise.
      - Time is either in 12-hour format with the indicator(AM/PM) in UPPERCASE or 24-hour optionally having the seconds.
      - Both date and time should be close to each other for example '12/12/2012 at 03:40:30 hrs'.
- **Amount:**
      - The actual amount being transferred.
      - The amount should have a wide range i.e. from millions to tens but not zeros.
      - Directly affects the transaction cost
- **Balance:**
      - The remaining amount in the sending account.
      - Optionally include this.
      - Similar to transaction codes, place it anywhere within the text but should be indicated as the balance in order not to confuse the model as an amount or transaction cost.
      - Occasionally include 0 balances.
- **Cost:**
      - The cost of transferring the amount.
      - Optionally included.
      - The cost varies with the amount being sent, if the amount is higher, the cost should also be high.
      - Should be at a reasonable range for example 3-digit amounts incur 1-digit costs
### 2. Metadata generation
- This is simply what's being extracted from the text.
- Each extracted value should be identical to what's in the text i.e. If it's UPPERCASE in text, it should be UPPERCASE in value.
- Each value MUST be found in the generated text.

## Example text:
The following are some examples on how texts look like, you are not to copy-paste them but to learn from them:
1. Confirmed. F8SFD6FW95 successfully withdrawn Kshs 3000.00 from ABSA BANK account 1234 with M-Pesa ref.5SRE6WE68F at 02-20-2025 3:15 PM with a transaction cost ksh 20.00. New balance is ksh 340.00.
2. Dear John, your account ending with ******45678 has received a debit of Ksh. 3000 on 31/12/2005 at 12:30:03 hrs.
3. R98HTG56FHG56 successfully received ksh 500,000.00 from SOME BANK account xyz. New balance is ksh 600,000.00. You can check your balance at https://check-balance.com/1seoijr
"""

In [ ]:
client = Client()
transactions: list[Transaction] = []
for _ in range(10):
  response = client.interactions.create(
      model='gemini-3.5-flash-lite',
      input=system_prompt,
      response_format={
        'type': 'text',
        "schema": TransactionsList.model_json_schema(),
        'mime_type': 'application/json'
      }
  )
  data: TransactionsList = TransactionsList.model_validate_json(response.output_text)

  transactions.extend(data.transactions)

In [ ]:

with open('data.json', 'r') as f:
  saved_data = json.load(f)

saved_data.append([transaction.model_dump() for transaction in transactions])

with open('data.json', 'w') as f:
  json.dump(saved_data, f, indent=4)